# 📚 Mise à jour des prix de vente Odoo — DSM Librairie

## Mode d'emploi (3 minutes)

1. Cliquez sur le bouton **▶** à gauche de la cellule de code ci-dessous
2. Une fenêtre d'envoi apparaît : choisissez votre fichier CSV
   - **`prix_vente_test.csv`** pour tester sur 5 articles d'abord ✅ (recommandé)
   - **`prix_vente_2026_2027.csv`** pour le tarif complet (4 103 articles)
3. Appuyez sur **Entrée** pour accepter l'URL et la base proposées
4. Entrez votre **login** et **mot de passe** Odoo
5. Lisez le **rapport de simulation** (rien n'est encore modifié)
6. Tapez **oui** pour appliquer, ou autre chose pour annuler

⚠️ Faites d'abord le test avec 5 articles, vérifiez dans Odoo, puis relancez (▶) avec le fichier complet.

In [ ]:
# -*- coding: utf-8 -*-
import csv, getpass, sys, xmlrpc.client

ODOO_URL = "http://94.130.90.253:9069"
ODOO_DB = "agora-prod"
TAILLE_LOT = 500

# ── 1. Fichier CSV ──
try:
    from google.colab import files
    print("➜ Envoyez votre fichier CSV (prix_vente_test.csv ou prix_vente_2026_2027.csv)")
    envoyes = files.upload()
    contenu = next(iter(envoyes.values())).decode("utf-8-sig")
except ImportError:
    with open(input("Chemin du fichier CSV : ").strip().strip('"'), encoding="utf-8-sig") as f:
        contenu = f.read()

entrees = []
for ligne in csv.reader(contenu.splitlines(), delimiter=";"):
    if len(ligne) < 2 or not ligne[0].strip().isdigit():
        continue
    entrees.append((ligne[0].strip(), float(ligne[1].strip().replace(",", "."))))
if not entrees:
    sys.exit("Erreur : aucune ligne exploitable (format attendu : barcode;prix_vente)")
print(f"✔ Fichier lu : {len(entrees)} articles")

# ── 2. Connexion Odoo (XML-RPC) ──
url = input(f"URL Odoo [{ODOO_URL}] : ").strip() or ODOO_URL
db = input(f"Base de données [{ODOO_DB}] : ").strip() or ODOO_DB
login = input("Login Odoo : ").strip()
mdp = getpass.getpass("Mot de passe Odoo : ")

common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common", allow_none=True)
uid = common.authenticate(db, login, mdp, {})
if not uid:
    sys.exit("Erreur : authentification refusée — vérifiez login/mot de passe")
modeles = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)
print(f"✔ Connecté à {url} (base {db}, utilisateur #{uid})")

def rpc(methode, *args, **kwargs):
    return modeles.execute_kw(db, uid, mdp, "product.template", methode, list(args), kwargs)

# ── 3. Recherche des produits ──
codes = [c for c, _ in entrees]
correspondance = {}
for debut in range(0, len(codes), TAILLE_LOT):
    lot = codes[debut : debut + TAILLE_LOT]
    for p in rpc("search_read", [["barcode", "in", lot]], fields=["id", "barcode", "name", "list_price"]):
        correspondance.setdefault(str(p["barcode"]), []).append(p["id"])
    print(f"  … recherche {min(debut + TAILLE_LOT, len(codes))}/{len(codes)}")

maj, introuvables = {}, []
for code, prix in entrees:
    ids = correspondance.get(code)
    if ids:
        maj.setdefault(round(prix, 2), []).extend(ids)
    else:
        introuvables.append(code)

nb = sum(len(v) for v in maj.values())
print("\n════════ RAPPORT (SIMULATION — rien n'est encore modifié) ════════")
print(f"  Produits trouvés dans Odoo : {nb}")
print(f"  Codes-barres introuvables  : {len(introuvables)}")
if introuvables:
    for c in introuvables[:30]:
        print(f"    - {c}")
    if len(introuvables) > 30:
        print(f"    … et {len(introuvables) - 30} autres")
if not maj:
    sys.exit("Rien à mettre à jour.")

# Aperçu avant/après
apercu_ids = [ids[0] for ids in list(maj.values())[:5]]
prix_par_id = {pid: prix for prix, ids in maj.items() for pid in ids}
print("\n  Aperçu (5 premiers) :")
for p in rpc("read", apercu_ids, fields=["name", "barcode", "list_price"]):
    print(f"    {p['barcode']} | {p['name'][:45]:45s} | {p['list_price']:.2f} → {prix_par_id[p['id']]:.2f}")

# ── 4. Confirmation puis application ──
reponse = input(f"\n➜ Appliquer ces {nb} changements ? (tapez oui pour confirmer) : ").strip().lower()
if reponse in ("oui", "o", "yes", "y"):
    total = 0
    for prix, ids in sorted(maj.items()):
        for debut in range(0, len(ids), TAILLE_LOT):
            lot = ids[debut : debut + TAILLE_LOT]
            rpc("write", lot, {"list_price": prix})
            total += len(lot)
            print(f"  ✔ {total}/{nb} produits mis à jour…")
    print(f"\n✔ TERMINÉ : {total} produit(s) mis à jour.")
else:
    print("Abandon — aucun prix modifié.")